In [48]:
import pandas as pd
import warnings
warnings.simplefilter('ignore')
print('succes')

succes


In [49]:
import pyodbc

DBSDM  = {
    'servername' : r'LAPTOP-28CE8V5M\SQLEXPRESS',
    'database' : 'Source Data Model'
}

DBETL  = {
    'servername' : r'LAPTOP-28CE8V5M\SQLEXPRESS',
    'database' : 'ETL'
}

export_conn_sdm = pyodbc.connect(
    'DRIVER={SQL SERVER};SERVER=' +
    DBSDM['servername'] + 
    ';DATABASE=' + 
    DBSDM['database'] + 
    ';Trusted_Connection=yes'
)

export_conn_etl = pyodbc.connect(
    'DRIVER={SQL SERVER};SERVER=' +
    DBETL['servername'] + 
    ';DATABASE=' + 
    DBETL['database'] + 
    ';Trusted_Connection=yes'
)

export_cursor_sdm = export_conn_sdm.cursor()
export_cursor_etl = export_conn_etl.cursor()

print('done')

done


In [50]:
export_cursor_etl.execute("EXEC sp_MSforeachtable 'ALTER TABLE ? NOCHECK CONSTRAINT ALL'")

export_cursor_etl.execute("SELECT TABLE_NAME FROM INFORMATION_SCHEMA.TABLES WHERE TABLE_TYPE = 'BASE TABLE'")
tables = export_cursor_etl.fetchall()

for table in tables:
    table_name = table[0]
    export_cursor_etl.execute(f"DELETE FROM {table_name}")
    print(f"Emptied table: {table_name}")

export_cursor_etl.execute("EXEC sp_MSforeachtable 'ALTER TABLE ? WITH CHECK CHECK CONSTRAINT ALL'")

export_cursor_etl.commit()

Emptied table: product_dim
Emptied table: sales_staff
Emptied table: order_details_dim
Emptied table: return_reason
Emptied table: returned_item
Emptied table: course
Emptied table: training


In [51]:
product_dimensie = pd.read_sql("SELECT * FROM product", export_conn_sdm)
product_type_dimensie = pd.read_sql("SELECT * FROM product_type", export_conn_sdm)
product_line_dimensie = pd.read_sql("SELECT * FROM product_line", export_conn_sdm)
product_line_type_dimensie = pd.merge(product_type_dimensie, product_line_dimensie, 
                     left_on=["PRODUCT_LINE_CODE"], 
                     right_on=["PRODUCT_LINE_CODE"], 
                     how="outer")
product_complete_dimensie = pd.merge(product_dimensie, product_line_type_dimensie, 
                     left_on=["PRODUCT_TYPE_CODE"], 
                     right_on=["PRODUCT_TYPE_CODE"], 
                     how="outer")

for index, row in product_complete_dimensie.iterrows():
    try:
        query = f"INSERT INTO product_dim VALUES ({row['PRODUCT_NUMBER']}, '{row['INTRODUCTION_DATE']}', {row['PRODUCT_TYPE_CODE']}, {row['PRODUCTION_COST']}, {row['MARGIN']}, '{row['LANGUAGE']}', '{row['PRODUCT_NAME'].replace("'", "''")}', '{row['PRODUCT_TYPE_EN']}', {row['PRODUCT_LINE_CODE']}, '{row['PRODUCT_LINE_EN']}')"
        export_cursor_etl.execute(query)
    except pyodbc.Error:
        print(query)

export_conn_etl.commit()

sales_staff_dimensie = pd.read_sql("SELECT * FROM sales_staff", export_conn_sdm)

for index, row in sales_staff_dimensie.iterrows():
    try:
        query = f"INSERT INTO sales_staff VALUES ({row['SALES_STAFF_CODE']}, '{row['FIRST_NAME'].replace("'", "''")}', '{row['LAST_NAME'].replace("'", "''")}', '{row['POSITION_EN'].replace("'", "''")}', '{row['WORK_PHONE']}', {row['EXTENSION'] if (row['EXTENSION'] > 0) else 'null'}, '{row['FAX']}', '{row['EMAIL']}', '{row['DATE_HIRED']}', {row['SALES_BRANCH_CODE']} , {row['MANAGER_CODE'] if (row['MANAGER_CODE'] != None) else 'null'})"
        export_cursor_etl.execute(query)
    except pyodbc.Error:
        print(query)

export_conn_etl.commit()

order_details_dimensie = pd.read_sql("SELECT * FROM order_details", export_conn_sdm)
order_header_dimensie = pd.read_sql("SELECT * FROM order_header", export_conn_sdm)
order_details_dim_dimensie = pd.merge(order_details_dimensie, order_header_dimensie, 
                     left_on=["ORDER_NUMBER"], 
                     right_on=["ORDER_NUMBER"], 
                     how="outer")

for index, row in order_details_dim_dimensie.iterrows():
    try:
        query = f"INSERT INTO order_details_dim VALUES ({row['ORDER_DETAIL_CODE']}, {row['ORDER_NUMBER']}, {row['PRODUCT_NUMBER']}, {row['QUANTITY']}, {row['UNIT_COST']}, {row['UNIT_PRICE']}, {row['UNIT_SALE_PRICE']}, '{row['RETAILER_NAME'].replace("'", "''")}', {row['RETAILER_SITE_CODE']}, {row['RETAILER_CONTACT_CODE']}, {row['SALES_STAFF_CODE']}, {row['SALES_BRANCH_CODE']}, '{row['ORDER_DATE']}', {row['ORDER_METHOD_CODE']})"
        export_cursor_etl.execute(query)
    except pyodbc.Error:
        print(query)

export_conn_etl.commit()

return_reason_dimensie = pd.read_sql("SELECT * FROM return_reason", export_conn_sdm)

for index, row in return_reason_dimensie.iterrows():
    try:
        query = f"INSERT INTO return_reason VALUES ({row['RETURN_REASON_CODE']}, '{row['RETURN_DESCRIPTION_EN'].replace("'", "''")}')"
        export_cursor_etl.execute(query)
    except pyodbc.Error:
        print(query)

export_conn_etl.commit()

returned_item_dimensie = pd.read_sql("SELECT * FROM returned_item", export_conn_sdm)
order_details_dimensie = pd.read_sql("SELECT * FROM order_details", export_conn_sdm)
returned_item_complete_dimensie = pd.merge(returned_item_dimensie, order_details_dimensie, 
                     left_on=["ORDER_DETAIL_CODE"], 
                     right_on=["ORDER_DETAIL_CODE"], 
                     how="inner")

for index, row in returned_item_complete_dimensie.iterrows():
    try:
        query = f"INSERT INTO returned_item VALUES ({row['RETURN_CODE']}, '{row['RETURN_DATE']}', {row['ORDER_DETAIL_CODE']}, {row['RETURN_REASON_CODE']}, {row['RETURN_QUANTITY']}, {row['PRODUCT_NUMBER']})"
        export_cursor_etl.execute(query)
    except pyodbc.Error:
        print(query)

export_conn_etl.commit()

course_dimensie = pd.read_sql("SELECT * FROM course", export_conn_sdm)

for index, row in course_dimensie.iterrows():
    try:
        query = f"INSERT INTO course VALUES ({row['COURSE_CODE']}, '{row['COURSE_DESCRIPTION'].replace("'", "''")}')"
        export_cursor_etl.execute(query)
    except pyodbc.Error:
        print(query)

export_conn_etl.commit()

training_dimensie = pd.read_sql("SELECT * FROM training", export_conn_sdm)

for index, row in training_dimensie.iterrows():
    try:
        query = f"INSERT INTO training VALUES ({row['YEAR']}, {row['SALES_STAFF_CODE']}, {row['COURSE_CODE']})"
        export_cursor_etl.execute(query)
    except pyodbc.Error:
        print(query)

export_conn_etl.commit()
export_conn_etl.close()
export_conn_sdm.close()